In [ ]:
#Controlador em C++
#include <Servo.h>

Servo arm1;
Servo arm2;
Servo base;

const int ARM1_PIN = 10;
const int ARM2_PIN = 11;
const int BASE_PIN = 9;

const int ARM_REL_1 = 20;
const int ARM_REL_2 = 160;
const int ARM_HOLD_1 = 90;
const int ARM_HOLD_2 = 90;

const int BASE_STOP = 90;
const int BASE_CW = 0;
const int BASE_CCW = 180;

String inputLine = "";

void moveArmRelease() {
  arm1.write(ARM_REL_1);
  arm2.write(ARM_REL_2);
  delay(500);
}

void moveArmHold() {
  arm1.write(ARM_HOLD_1);
  arm2.write(ARM_HOLD_2);
  delay(500);
}

void flipCube() {
  arm1.write(180);
  arm2.write(0);
  delay(800);
  arm1.write(20);
  arm2.write(160);
  delay(800);
}

void rotateBase(int direction, int durationMs) {
  base.write(direction);
  delay(durationMs);
  base.write(BASE_STOP);
  delay(100);
}

void executeMove(char mv) {
  switch (mv) {
    case 'U':
      moveArmRelease();
      rotateBase(BASE_CW, 350);
      Serial.println("DONE");
      break;

    case 'u':
      moveArmRelease();
      rotateBase(BASE_CCW, 350);
      Serial.println("DONE");
      break;

    case 'D':
      moveArmRelease();
      rotateBase(BASE_CW, 700);
      Serial.println("DONE");
      break;

    case 'd':
      moveArmRelease();
      rotateBase(BASE_CCW, 700);
      Serial.println("DONE");
      break;

    case 'R':
      moveArmHold();
      rotateBase(BASE_CW, 350);
      Serial.println("DONE");
      break;

    case 'r':
      moveArmHold();
      rotateBase(BASE_CCW, 350);
      Serial.println("DONE");
      break;

    case 'L':
      moveArmHold();
      rotateBase(BASE_CW, 700);
      Serial.println("DONE");
      break;

    case 'l':
      moveArmHold();
      rotateBase(BASE_CCW, 700);
      Serial.println("DONE");
      break;

    case 'F':
      flipCube();
      Serial.println("DONE");
      break;

    case 'B':
      flipCube();
      flipCube();
      flipCube();
      Serial.println("DONE");
      break;

    default:
      Serial.println("ERROR");
      break;
  }
}

void handleCommand(String cmd) {
  cmd.trim();

  if (cmd == "PC_READY") {
    Serial.println("READY");
  }
  else if (cmd == "NEXT") { // Este comando não será usado pela nova lógica Python, mas pode ser mantido.
    Serial.println("SCAN");
  }
  else if (cmd == "FINISH") {
    moveArmRelease();
    Serial.println("DONE");
  }
  else if (cmd == "RESET") {
    moveArmRelease();
    base.write(BASE_STOP);
    Serial.println("READY");
  }
  else if (cmd.startsWith("MOVE ")) {
    char mv = cmd.charAt(5);
    executeMove(mv);
  }
  else if (cmd == "ERROR") { // Adicionei tratamento para o comando ERROR, caso o Python o envie
    Serial.println("DONE");
  }
}

void setup() {
  Serial.begin(9600);

  arm1.attach(ARM1_PIN);
  arm2.attach(ARM2_PIN);
  base.attach(BASE_PIN);

  moveArmRelease(); // Garante que os braços estejam na posição de soltura
  base.write(BASE_STOP); // Garante que a base esteja parada

  delay(1000); // Pequena pausa para estabilização
  Serial.println("READY"); // Informa ao PC que o Arduino está pronto para comandos
}

void loop() {
  while (Serial.available()) {
    char c = Serial.read();

    if (c == '\n') {
      handleCommand(inputLine);
      inputLine = "";
    } else if (c != '\r') {
      inputLine += c;
    }
  }
}
